# 🔬 Fine-Tuning Notebook: GPT-2 & DistilBERT on AG News
> **Methods:** Full-Phase Fine-Tuning · LoRA · Adapter  
> **Dataset:** AG News (4-class text classification)  
> **Server:** Run on GPU server via SSH  
> Results feed directly into the comparison Excel sheet.

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install transformers datasets evaluate peft accelerate
!pip install adapter-transformers scikit-learn tqdm openpyxl
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118

## ⚙️ Step 2 — Imports & Config

In [ ]:
import os, time, json, torch
import numpy as np
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from datasets import load_dataset
from evaluate import load as load_metric
from peft import get_peft_model, LoraConfig, TaskType
import torch.nn as nn

# ── Config ──────────────────────────────────────────────
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_LABELS  = 4          # AG News has 4 classes
MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS      = 3
LR          = 2e-5
SEED        = 42
RESULTS_DIR = './results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Using device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 📂 Step 3 — Load AG News Dataset

In [ ]:
dataset    = load_dataset('ag_news')
train_data = dataset['train'].shuffle(seed=SEED).select(range(10000))  # 10k for speed
test_data  = dataset['test'].shuffle(seed=SEED).select(range(2000))

print(f'Train size: {len(train_data)}')
print(f'Test  size: {len(test_data)}')
print(f'Classes   : {train_data.features["label"].names}')

## 🔧 Step 4 — Helper Functions

In [ ]:
metric = load_metric('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import precision_recall_fscore_support
    acc = metric.compute(predictions=preds, references=labels)['accuracy']
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

def measure_latency(model, tokenizer, n=100):
    """Average inference latency over n samples."""
    model.eval()
    sample_texts = [test_data[i]['text'] for i in range(n)]
    inputs = tokenizer(sample_texts, return_tensors='pt',
                       truncation=True, padding=True,
                       max_length=MAX_LEN).to(DEVICE)
    with torch.no_grad():
        start = time.time()
        for _ in range(n):
            enc = tokenizer(test_data[0]['text'], return_tensors='pt',
                            truncation=True, max_length=MAX_LEN).to(DEVICE)
            model(**enc)
    return (time.time() - start) / n

def get_gpu_memory():
    """Peak GPU memory in MB."""
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / 1024**2
    return 0.0

def get_throughput(model, tokenizer, n=500):
    """Samples per second."""
    model.eval()
    texts  = [test_data[i % len(test_data)]['text'] for i in range(n)]
    inputs = tokenizer(texts, return_tensors='pt', truncation=True,
                       padding=True, max_length=MAX_LEN).to(DEVICE)
    start  = time.time()
    with torch.no_grad():
        model(**inputs)
    return n / (time.time() - start)

results = []  # will collect all rows

---
## 🤖 MODEL 1: GPT-2

### 1A — GPT-2 Full-Phase Fine-Tuning

In [ ]:
MODEL_NAME = 'gpt2'
tokenizer_gpt2 = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token

def tokenize_gpt2(batch):
    return tokenizer_gpt2(batch['text'], truncation=True,
                          max_length=MAX_LEN, padding='max_length')

train_tok_gpt2 = train_data.map(tokenize_gpt2, batched=True)
test_tok_gpt2  = test_data.map(tokenize_gpt2,  batched=True)

torch.cuda.reset_peak_memory_stats()
model_gpt2_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True).to(DEVICE)
model_gpt2_full.config.pad_token_id = tokenizer_gpt2.pad_token_id

args_gpt2_full = TrainingArguments(
    output_dir='./gpt2_full', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR, evaluation_strategy='epoch',
    save_strategy='no', seed=SEED, report_to='none',
    fp16=torch.cuda.is_available())

trainer_gpt2_full = Trainer(
    model=model_gpt2_full, args=args_gpt2_full,
    train_dataset=train_tok_gpt2, eval_dataset=test_tok_gpt2,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer_gpt2))

t_start = time.time()
trainer_gpt2_full.train()
train_time = time.time() - t_start

eval_out  = trainer_gpt2_full.evaluate()
latency   = measure_latency(model_gpt2_full, tokenizer_gpt2)
gpu_mem   = get_gpu_memory()
throughput= get_throughput(model_gpt2_full, tokenizer_gpt2)
energy    = train_time * 0.000071  # ~71W GPU average -> kWh

results.append({
    'Model': 'GPT-2', 'Fine-Tuning Method': 'Full-phase Fine-Tuning',
    'Accuracy': eval_out['eval_accuracy'],
    'Training Time (s)': round(train_time, 3),
    'Energy Consumption (kWh)': round(energy, 6),
    'Latency per sample (s)': round(latency, 6),
    'GPU Memory Usage (MB)': round(gpu_mem, 3),
    'Precision': round(eval_out['eval_precision'], 6),
    'Recall': round(eval_out['eval_recall'], 6),
    'F1 Score': round(eval_out['eval_f1'], 6),
    'Throughput': round(throughput, 3)
})
print('GPT-2 Full-phase done:', results[-1])

### 1B — GPT-2 LoRA Fine-Tuning

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

torch.cuda.reset_peak_memory_stats()
base_gpt2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True).to(DEVICE)
base_gpt2.config.pad_token_id = tokenizer_gpt2.pad_token_id

lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
    target_modules=['c_attn'], lora_dropout=0.1, bias='none')
model_gpt2_lora = get_peft_model(base_gpt2, lora_cfg)
model_gpt2_lora.print_trainable_parameters()

args_lora = TrainingArguments(
    output_dir='./gpt2_lora', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=3e-4, evaluation_strategy='epoch',
    save_strategy='no', seed=SEED, report_to='none',
    fp16=torch.cuda.is_available())

trainer_lora = Trainer(
    model=model_gpt2_lora, args=args_lora,
    train_dataset=train_tok_gpt2, eval_dataset=test_tok_gpt2,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer_gpt2))

t_start = time.time()
trainer_lora.train()
train_time = time.time() - t_start

eval_out   = trainer_lora.evaluate()
latency    = measure_latency(model_gpt2_lora, tokenizer_gpt2)
gpu_mem    = get_gpu_memory()
throughput = get_throughput(model_gpt2_lora, tokenizer_gpt2)
energy     = train_time * 0.000071

results.append({
    'Model': 'GPT-2', 'Fine-Tuning Method': 'LORA',
    'Accuracy': eval_out['eval_accuracy'],
    'Training Time (s)': round(train_time, 3),
    'Energy Consumption (kWh)': round(energy, 6),
    'Latency per sample (s)': round(latency, 6),
    'GPU Memory Usage (MB)': round(gpu_mem, 3),
    'Precision': round(eval_out['eval_precision'], 6),
    'Recall': round(eval_out['eval_recall'], 6),
    'F1 Score': round(eval_out['eval_f1'], 6),
    'Throughput': round(throughput, 3)
})
print('GPT-2 LoRA done:', results[-1])

### 1C — GPT-2 Adapter Fine-Tuning

In [ ]:
class AdapterLayer(nn.Module):
    def __init__(self, d_model, bottleneck=64):
        super().__init__()
        self.down = nn.Linear(d_model, bottleneck)
        self.act  = nn.GELU()
        self.up   = nn.Linear(bottleneck, d_model)
    def forward(self, x):
        return x + self.up(self.act(self.down(x)))

torch.cuda.reset_peak_memory_stats()
model_gpt2_adapter = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    ignore_mismatched_sizes=True).to(DEVICE)
model_gpt2_adapter.config.pad_token_id = tokenizer_gpt2.pad_token_id

# Freeze all params, add adapters to each transformer block
for param in model_gpt2_adapter.parameters():
    param.requires_grad = False

d_model = model_gpt2_adapter.config.n_embd
for block in model_gpt2_adapter.transformer.h:
    block.adapter = AdapterLayer(d_model).to(DEVICE)
    orig_forward = block.forward
    def make_forward(blk):
        orig = blk.__class__.forward
        def new_forward(self, *args, **kwargs):
            out = orig(self, *args, **kwargs)
            hidden = out[0]
            adapted = self.adapter(hidden)
            return (adapted,) + out[1:]
        return new_forward
    import types
    block.forward = types.MethodType(make_forward(block), block)

# Unfreeze classifier + adapters
for name, param in model_gpt2_adapter.named_parameters():
    if 'adapter' in name or 'score' in name:
        param.requires_grad = True

trainable = sum(p.numel() for p in model_gpt2_adapter.parameters() if p.requires_grad)
print(f'Trainable adapter params: {trainable:,}')

args_adapter = TrainingArguments(
    output_dir='./gpt2_adapter', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=1e-4, evaluation_strategy='epoch',
    save_strategy='no', seed=SEED, report_to='none',
    fp16=torch.cuda.is_available())

trainer_adapter = Trainer(
    model=model_gpt2_adapter, args=args_adapter,
    train_dataset=train_tok_gpt2, eval_dataset=test_tok_gpt2,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer_gpt2))

t_start = time.time()
trainer_adapter.train()
train_time = time.time() - t_start

eval_out   = trainer_adapter.evaluate()
latency    = measure_latency(model_gpt2_adapter, tokenizer_gpt2)
gpu_mem    = get_gpu_memory()
throughput = get_throughput(model_gpt2_adapter, tokenizer_gpt2)
energy     = train_time * 0.000071

results.append({
    'Model': 'GPT-2', 'Fine-Tuning Method': 'Adapter',
    'Accuracy': eval_out['eval_accuracy'],
    'Training Time (s)': round(train_time, 3),
    'Energy Consumption (kWh)': round(energy, 6),
    'Latency per sample (s)': round(latency, 6),
    'GPU Memory Usage (MB)': round(gpu_mem, 3),
    'Precision': round(eval_out['eval_precision'], 6),
    'Recall': round(eval_out['eval_recall'], 6),
    'F1 Score': round(eval_out['eval_f1'], 6),
    'Throughput': round(throughput, 3)
})
print('GPT-2 Adapter done:', results[-1])

---
## 🤖 MODEL 2: DistilBERT

### 2A — DistilBERT Full-Phase Fine-Tuning

In [ ]:
MODEL_NAME_DIST = 'distilbert-base-uncased'
tokenizer_dist  = AutoTokenizer.from_pretrained(MODEL_NAME_DIST)

def tokenize_dist(batch):
    return tokenizer_dist(batch['text'], truncation=True,
                          max_length=MAX_LEN, padding='max_length')

train_tok_dist = train_data.map(tokenize_dist, batched=True)
test_tok_dist  = test_data.map(tokenize_dist,  batched=True)

torch.cuda.reset_peak_memory_stats()
model_dist_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DIST, num_labels=NUM_LABELS).to(DEVICE)

args_dist_full = TrainingArguments(
    output_dir='./dist_full', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR, evaluation_strategy='epoch',
    save_strategy='no', seed=SEED, report_to='none',
    fp16=torch.cuda.is_available())

trainer_dist_full = Trainer(
    model=model_dist_full, args=args_dist_full,
    train_dataset=train_tok_dist, eval_dataset=test_tok_dist,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer_dist))

t_start = time.time()
trainer_dist_full.train()
train_time = time.time() - t_start

eval_out   = trainer_dist_full.evaluate()
latency    = measure_latency(model_dist_full, tokenizer_dist)
gpu_mem    = get_gpu_memory()
throughput = get_throughput(model_dist_full, tokenizer_dist)
energy     = train_time * 0.000071

results.append({
    'Model': 'DistilBERT', 'Fine-Tuning Method': 'Full-phase Fine-Tuning',
    'Accuracy': eval_out['eval_accuracy'],
    'Training Time (s)': round(train_time, 3),
    'Energy Consumption (kWh)': round(energy, 6),
    'Latency per sample (s)': round(latency, 6),
    'GPU Memory Usage (MB)': round(gpu_mem, 3),
    'Precision': round(eval_out['eval_precision'], 6),
    'Recall': round(eval_out['eval_recall'], 6),
    'F1 Score': round(eval_out['eval_f1'], 6),
    'Throughput': round(throughput, 3)
})
print('DistilBERT Full-phase done:', results[-1])

### 2B — DistilBERT LoRA

In [ ]:
torch.cuda.reset_peak_memory_stats()
base_dist = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DIST, num_labels=NUM_LABELS).to(DEVICE)

lora_cfg_dist = LoraConfig(
    task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
    target_modules=['q_lin', 'k_lin'], lora_dropout=0.1, bias='none')
model_dist_lora = get_peft_model(base_dist, lora_cfg_dist)
model_dist_lora.print_trainable_parameters()

args_dist_lora = TrainingArguments(
    output_dir='./dist_lora', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=3e-4, evaluation_strategy='epoch',
    save_strategy='no', seed=SEED, report_to='none',
    fp16=torch.cuda.is_available())

trainer_dist_lora = Trainer(
    model=model_dist_lora, args=args_dist_lora,
    train_dataset=train_tok_dist, eval_dataset=test_tok_dist,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer_dist))

t_start = time.time()
trainer_dist_lora.train()
train_time = time.time() - t_start

eval_out   = trainer_dist_lora.evaluate()
latency    = measure_latency(model_dist_lora, tokenizer_dist)
gpu_mem    = get_gpu_memory()
throughput = get_throughput(model_dist_lora, tokenizer_dist)
energy     = train_time * 0.000071

results.append({
    'Model': 'DistilBERT', 'Fine-Tuning Method': 'LORA',
    'Accuracy': eval_out['eval_accuracy'],
    'Training Time (s)': round(train_time, 3),
    'Energy Consumption (kWh)': round(energy, 6),
    'Latency per sample (s)': round(latency, 6),
    'GPU Memory Usage (MB)': round(gpu_mem, 3),
    'Precision': round(eval_out['eval_precision'], 6),
    'Recall': round(eval_out['eval_recall'], 6),
    'F1 Score': round(eval_out['eval_f1'], 6),
    'Throughput': round(throughput, 3)
})
print('DistilBERT LoRA done:', results[-1])

### 2C — DistilBERT Adapter

In [ ]:
torch.cuda.reset_peak_memory_stats()
model_dist_adapter = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME_DIST, num_labels=NUM_LABELS).to(DEVICE)

for param in model_dist_adapter.parameters():
    param.requires_grad = False

d_model_dist = model_dist_adapter.config.hidden_size
for layer in model_dist_adapter.distilbert.transformer.layer:
    layer.adapter = AdapterLayer(d_model_dist).to(DEVICE)

for name, param in model_dist_adapter.named_parameters():
    if 'adapter' in name or 'classifier' in name or 'pre_classifier' in name:
        param.requires_grad = True

args_dist_adapter = TrainingArguments(
    output_dir='./dist_adapter', num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=1e-4, evaluation_strategy='epoch',
    save_strategy='no', seed=SEED, report_to='none',
    fp16=torch.cuda.is_available())

trainer_dist_adapter = Trainer(
    model=model_dist_adapter, args=args_dist_adapter,
    train_dataset=train_tok_dist, eval_dataset=test_tok_dist,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer_dist))

t_start = time.time()
trainer_dist_adapter.train()
train_time = time.time() - t_start

eval_out   = trainer_dist_adapter.evaluate()
latency    = measure_latency(model_dist_adapter, tokenizer_dist)
gpu_mem    = get_gpu_memory()
throughput = get_throughput(model_dist_adapter, tokenizer_dist)
energy     = train_time * 0.000071

results.append({
    'Model': 'DistilBERT', 'Fine-Tuning Method': 'Adapter',
    'Accuracy': eval_out['eval_accuracy'],
    'Training Time (s)': round(train_time, 3),
    'Energy Consumption (kWh)': round(energy, 6),
    'Latency per sample (s)': round(latency, 6),
    'GPU Memory Usage (MB)': round(gpu_mem, 3),
    'Precision': round(eval_out['eval_precision'], 6),
    'Recall': round(eval_out['eval_recall'], 6),
    'F1 Score': round(eval_out['eval_f1'], 6),
    'Throughput': round(throughput, 3)
})
print('DistilBERT Adapter done:', results[-1])

## 💾 Step 5 — Save Results to CSV & Excel

In [ ]:
df = pd.DataFrame(results)
print('\n=== ALL RESULTS ===')
print(df.to_string(index=False))

# Save CSV
csv_path = os.path.join(RESULTS_DIR, 'finetuning_results_ag_news.csv')
df.to_csv(csv_path, index=False)
print(f'\nSaved CSV → {csv_path}')

# Save Excel with formatting
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = Workbook()
for model_name, grp in df.groupby('Model'):
    ws = wb.create_sheet(model_name)
    headers = list(df.columns)
    for ci, h in enumerate(headers, 1):
        c = ws.cell(row=1, column=ci, value=h)
        c.font = Font(bold=True, color='FFFFFF')
        c.fill = PatternFill('solid', start_color='1F3864', end_color='1F3864')
        c.alignment = Alignment(horizontal='center', wrap_text=True)
    for ri, row in enumerate(grp.itertuples(index=False), 2):
        for ci, val in enumerate(row, 1):
            ws.cell(row=ri, column=ci, value=val)

if 'Sheet' in wb.sheetnames:
    del wb['Sheet']
xl_path = os.path.join(RESULTS_DIR, 'finetuning_results_ag_news.xlsx')
wb.save(xl_path)
print(f'Saved Excel → {xl_path}')

## ✅ Expected Results (from your table)
```
GPT-2      | Full-phase | Acc=0.9459 | F1=0.9472 | Latency=0.001246s | GPU=2459MB | Throughput=211.65
GPT-2      | LoRA       | Acc=0.9309 | F1=0.9296 | Latency=0.001561s | GPU=1920MB | Throughput=267.47
GPT-2      | Adapter    | Acc=0.9224 | F1=0.9228 | Latency=0.001887s | GPU=1673MB | Throughput=186.75
DistilBERT | Full-phase | Acc=0.9459 | F1=0.9482 | Latency=0.001246s | GPU=2459MB | Throughput=369.19
DistilBERT | LoRA       | Acc=0.9309 | F1=0.9358 | Latency=0.001561s | GPU=1920MB | Throughput=452.42
DistilBERT | Adapter    | Acc=0.9224 | F1=0.9225 | Latency=0.001887s | GPU=1673MB | Throughput=391.18
```